# 01 - Ingesta Bronze: RUES (Registro Mercantil)

Trabajo Práctico 1 - Sección 3 (Fuente 1): Pipeline de ingesta PySpark

## Fuente

* **Dataset**: Personas Naturales, Personas Jurídicas y Entidades Sin Ánimo de Lucro (RUES)
* **API**: `https://www.datos.gov.co/resource/c82u-588k.json` (Socrata / SoQL)
* **Volumen total en la fuente**: 9.407.309 registros (verificado con `$select=count(*)`)
* **Destino**: `Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api`

## Estrategia de carga: por mes, con truncado idempotente

El dataset completo supera los 9 millones de registros (imprático de descargar por HTTP paginado en una sesión de clase), así que se carga **mes a mes**: se cambian las variables `ANIO` y `MES` y se filtra por `fecha_actualizacion` para traer solo ese rango.

Para que sea repetible sin duplicar datos, la escritura usa `replaceWhere`: si el mes seleccionado **ya se había cargado antes**, se trunca (borra) solo ese rango de fechas dentro de la tabla y se reemplaza con lo que traiga la API en esta ejecución; los demás meses ya cargados **no se tocan**. Así se puede ir acumulando la tabla mes por mes, sin reprocesar todo el histórico ni dejar duplicados si se vuelve a correr un mes ya cargado.

## Regla de inmutabilidad (Capa Bronze)

* No se renombran columnas: se conservan exactamente los nombres que entrega la API (`codigo_camara`, `razon_social`, etc.).
* No se hace *casting* destructivo: todas las columnas de RUES son de tipo `text` en el origen, así que se dejan como texto (no se convierten fechas a `DATE` ni números a `INT`/`DOUBLE`). Esa limpieza es tarea de la futura Capa Silver.
* Solo se agregan las columnas de auditoría obligatorias: `_ingested_at` y `_source`.

In [0]:
%python
import requests
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F

BASE_URL = "https://www.datos.gov.co/resource/c82u-588k.json"
LIMIT = 50000  # tamaño de página soportado por Socrata

# --- Cambia estos dos valores para cargar otro mes ---
ANIO = 2026
MES = 8
# ------------------------------------------------------


def rango_mes(anio, mes):
    inicio = datetime(anio, mes, 1)
    fin = datetime(anio + 1, 1, 1) if mes == 12 else datetime(anio, mes + 1, 1)
    return inicio.strftime("%Y/%m/%d"), fin.strftime("%Y/%m/%d")


# fecha_actualizacion llega como texto 'YYYY/MM/DD HH:MM:SS...', comparable como string
FECHA_INICIO, FECHA_FIN = rango_mes(ANIO, MES)
WHERE_CLAUSE = f"fecha_actualizacion >= '{FECHA_INICIO}' AND fecha_actualizacion < '{FECHA_FIN}'"

print(f"Fuente: {BASE_URL}")
print(f"Mes a cargar: {ANIO}-{MES:02d} ({FECHA_INICIO} a {FECHA_FIN})")
print(f"Filtro aplicado: {WHERE_CLAUSE}")

---

## Paso 1: Contar registros disponibles con el filtro aplicado

In [0]:
%python
def contar_registros(where=None):
    params = {"$select": "count(*)"}
    if where:
        params["$where"] = where
    resp = requests.get(BASE_URL, params=params)
    resp.raise_for_status()
    return int(resp.json()[0]["count"])


total_registros = contar_registros(WHERE_CLAUSE)
print(f"Total de registros de RUES para {ANIO}-{MES:02d}: {total_registros:,}")

---

## Paso 2: Descargar los datos paginando (HTTP GET + `$limit`/`$offset`)

In [0]:
%python
MAX_REGISTROS = total_registros


def descargar_datos(max_registros, where=None):
    registros = []
    offset = 0

    while offset < max_registros:
        pagina_limit = min(LIMIT, max_registros - offset)
        params = {"$limit": pagina_limit, "$offset": offset}
        if where:
            params["$where"] = where
        resp = requests.get(BASE_URL, params=params)
        resp.raise_for_status()
        pagina = resp.json()

        if not pagina:
            break

        registros.extend(pagina)
        offset += LIMIT
        print(f"Descargados {len(registros)} de {max_registros} registros...")

    return registros


registros = descargar_datos(MAX_REGISTROS, WHERE_CLAUSE)
df_pandas = pd.DataFrame(registros)
print(f"\nDataset descargado: {df_pandas.shape[0]} filas x {df_pandas.shape[1]} columnas")
df_pandas.head()

---

## Paso 3: Convertir a Spark DataFrame (sin transformar tipos) y agregar columnas de auditoría

In [0]:
%python
# Se respeta el tipo con el que Socrata entrega cada campo (todo texto en este dataset).
# No se hace ningún .astype()/cast manual: eso sería un casteo destructivo, prohibido en Bronze.
df_spark = spark.createDataFrame(df_pandas)

df_bronze = (
    df_spark
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source", F.lit(BASE_URL))
)

df_bronze.printSchema()
display(df_bronze.limit(10))

---

## Paso 4: Persistir en Delta Lake truncando solo el mes cargado

Si la tabla no existe todavía, se crea con el primer mes. Si ya existe, se usa `replaceWhere` con el mismo filtro de fecha del mes actual: eso borra e inserta de nuevo *solo* ese rango, dejando intactos los meses cargados en ejecuciones anteriores.

In [0]:
%python
TABLA_DESTINO = "Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api"

tabla_existe = spark.catalog.tableExists(TABLA_DESTINO)

writer = df_bronze.write.format("delta").mode("overwrite")

if tabla_existe:
    # Trunca solo el rango de fecha_actualizacion del mes actual; no toca otros meses ya cargados
    writer = writer.option("replaceWhere", WHERE_CLAUSE)
    print(f"Tabla existente: se trunca y recarga solo el rango {FECHA_INICIO} - {FECHA_FIN}")
else:
    writer = writer.option("overwriteSchema", "true")
    print("Tabla nueva: se crea con este primer mes cargado")

writer.saveAsTable(TABLA_DESTINO)

print(f"Tabla Delta actualizada: {TABLA_DESTINO} (mes {ANIO}-{MES:02d})")

In [0]:
%sql
-- Verificación rápida de la ingesta
DESCRIBE EXTENDED Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api;

In [0]:
%sql
SELECT COUNT(*) AS total_filas, MIN(_ingested_at) AS primera_carga, MAX(_ingested_at) AS ultima_carga
FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api;


In [0]:
%sql
-- Ver cuántas filas hay cargadas por cada mes (fecha_actualizacion) para confirmar
-- que se van acumulando meses sin duplicar
SELECT LEFT(fecha_actualizacion, 7) AS mes, COUNT(*) AS filas
FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api
GROUP BY LEFT(fecha_actualizacion, 7)
ORDER BY mes;

In [0]:
%sql
SELECT * FROM Datos_Empresas.bronze.DE_Semiestructurado_RegistroMercantil_Api LIMIT 10;

---

## Siguiente paso

Continuar con [`02_Ingesta_Bronze_TRM.ipynb`](02_Ingesta_Bronze_TRM.ipynb) para la fuente complementaria con carga incremental.